In [1]:
import os
import pandas as pd

folder_path = '/Users/anniewang/Desktop/MLLM-interpretability/MLLM-interpretability/data/KLAR/raw'
output_folder = '/Users/anniewang/Desktop/MLLM-interpretability/MLLM-interpretability/data/KLAR/raw/selected_rows'

os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(folder_path):
    if filename.endswith('.csv'):
        file_path = os.path.join(folder_path, filename)
        df = pd.read_csv(file_path, index_col=False)

        # randomly pick 1 row per "index" group
        selected_rows = (
            df.reset_index(drop=True)                      # clean pandas index only
            .groupby('q_index', group_keys=False, sort=False)
            .sample(n=1, random_state=42)
            .reset_index(drop=True)                      # clean output pandas index
        )

        output_file_path = os.path.join(output_folder, filename)
        selected_rows.to_csv(output_file_path, index=False)

In [3]:
#print unique values in the "relation" column for each target_lang
print(selected_rows['relation'].unique())

['instrument' 'religion' 'headquarters_location' 'developer'
 'applies_to_jurisdiction' 'country_of_citizenship' 'capital_of'
 'location_of_formation' 'native_language' 'official_language'
 'languages_spoken' 'place_of_death' 'manufacturer' 'continent' 'capital'
 'occupation' 'field_of_work' 'place_of_birth' 'owned_by'
 'language_of_work_or_name']


In [4]:
print(df[['q_index','prompt_id','q_id']].head())
print(selected_rows[['q_index','prompt_id','q_id']].head())

   q_index  prompt_id    q_id
0     4757          0  4757_0
1     4757          1  4757_1
2     4757          2  4757_2
3     4757          3  4757_3
4     4757          4  4757_4
   q_index  prompt_id    q_id
0     4757          1  4757_1
1     4758          3  4758_3
2     4759          1  4759_1
3     4760          0  4760_0
4     4761          0  4761_0


In [5]:
# combine a list of csv files into one csv file, and add a column "target_lang" to to be the filename without the extension, and save it as "combined.csv"
import glob
csv_files = glob.glob(os.path.join(output_folder, '*.csv'))
combined_df = pd.DataFrame()
for csv_file in csv_files:
    df = pd.read_csv(csv_file)
    target_lang = os.path.splitext(os.path.basename(csv_file))[0]
    df['target_lang'] = target_lang
    combined_df = pd.concat([combined_df, df], ignore_index=True)
    # insert a new column called "klar_id" to be the combination of "q_id" and "target_lang", separated by an underscore
    combined_df['klar_id'] = combined_df['target_lang'].astype(str) + '_' + combined_df['q_id'].astype(str)
combined_df.to_csv(os.path.join(output_folder, '../combined/klar_7.csv'), index=False)

In [6]:
unique_relations = combined_df['relation'].unique()
print(unique_relations)

['instrument' 'religion' 'headquarters_location' 'developer'
 'applies_to_jurisdiction' 'country_of_citizenship' 'capital_of'
 'location_of_formation' 'native_language' 'official_language'
 'languages_spoken' 'place_of_death' 'manufacturer' 'continent' 'capital'
 'occupation' 'field_of_work' 'place_of_birth' 'owned_by'
 'language_of_work_or_name']


In [7]:
cleaned_parts = []

for rel, rel_df in combined_df.groupby('relation'):
    langs = rel_df['target_lang'].unique()
    by_lang = rel_df.groupby('target_lang')

    # intersection across languages *within this relation*
    common_q = set.intersection(*[set(g['q_index']) for _, g in by_lang])

    cleaned_parts.append(rel_df[rel_df['q_index'].isin(common_q)])

cleaned_df = pd.concat(cleaned_parts, ignore_index=True)
cleaned_df.to_csv(os.path.join(output_folder, '../combined/klar_7_cleaned.csv'), index=False)

In [8]:
unique_relations = cleaned_df['relation'].unique()
print(unique_relations)

['applies_to_jurisdiction' 'capital' 'capital_of' 'continent'
 'country_of_citizenship' 'developer' 'field_of_work'
 'headquarters_location' 'instrument' 'language_of_work_or_name'
 'languages_spoken' 'location_of_formation' 'manufacturer'
 'native_language' 'occupation' 'official_language' 'owned_by'
 'place_of_birth' 'place_of_death' 'religion']


## Get JSON file for batch API

In [13]:
import pandas as pd
import json

dataset_name = "KLAR"

df = pd.read_csv(f"combined/{dataset_name}_7_cleaned.csv")

with open(f"{dataset_name}_batch_requests.jsonl", "w") as f:
    for _, row in df.iterrows():

        request = {
            "key": row["klar_id"],
            "request": {
                "contents": [
                    {
                        "role": "user",
                        "parts": [{"text": row["question"]}]
                    }
                ],
                "generation_config": {
                    "temperature": 0,
                    "top_p": 1,
                    "max_output_tokens": 64
                }
            }
        }

        f.write(json.dumps(request, ensure_ascii=False) + "\n")

In [11]:
import pandas as pd
dataset_name = "klar"
# load jsonl
df = pd.read_csv(f"combined/{dataset_name}_7_cleaned.csv")

# compute table
table = (
    df.groupby(["target_lang", "relation"])
      .size()
      .unstack(fill_value=0)
      .sort_index()
)

# add totals
table["Total"] = table.sum(axis=1)
table.loc["Total"] = table.sum()

stats_df = table.reset_index()
print(stats_df)

relation target_lang  applies_to_jurisdiction  capital  capital_of  continent  \
0                 en                       79      336         212        212   
1                 es                       79      336         212        212   
2                 fr                       79      336         212        212   
3                 he                       79      336         212        212   
4                 ja                       79      336         212        212   
5                 ko                       79      336         212        212   
6                 zh                       79      336         212        212   
7              Total                      553     2352        1484       1484   

relation  country_of_citizenship  developer  field_of_work  \
0                             60         76            167   
1                             60         76            167   
2                             60         76            167   
3                     

In [10]:
# print unique values in the "relation" column for each target_lang
print(cleaned_df.groupby('target_lang')['relation'].unique())
print(cleaned_df['relation'].unique())

target_lang
en    [applies_to_jurisdiction, capital, capital_of,...
es    [applies_to_jurisdiction, capital, capital_of,...
fr    [applies_to_jurisdiction, capital, capital_of,...
he    [applies_to_jurisdiction, capital, capital_of,...
ja    [applies_to_jurisdiction, capital, capital_of,...
ko    [applies_to_jurisdiction, capital, capital_of,...
zh    [applies_to_jurisdiction, capital, capital_of,...
Name: relation, dtype: object
['applies_to_jurisdiction' 'capital' 'capital_of' 'continent'
 'country_of_citizenship' 'developer' 'field_of_work'
 'headquarters_location' 'instrument' 'language_of_work_or_name'
 'languages_spoken' 'location_of_formation' 'manufacturer'
 'native_language' 'occupation' 'official_language' 'owned_by'
 'place_of_birth' 'place_of_death' 'religion']
